# Samsung Cloud Firmware Slice

Download/decrypt the full official Samsung firmware **inside Google Colab**, stream only the requested partition from the nested TAR package, decompress `.lz4` remotely, and download only the final payload to your computer.

The 8–12+ GB firmware never has to pass through your local connection.

In [ ]:
import os, subprocess, sys
repo = '/content/twrpdtgen-NEXT'
if not os.path.isdir(os.path.join(repo, '.git')):
    subprocess.run(['git', 'clone', '-q', 'https://github.com/RectifyData/twrpdtgen-NEXT.git', repo], check=True)
else:
    subprocess.run(['git', '-C', repo, 'pull', '--ff-only'], check=True)
os.chdir(repo)
subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', '-e', '.'], check=True)
print('twrpdtgen-NEXT cloud slicer ready')

In [ ]:
model = 'SM-G975F' #@param {type:'string'}
region = 'BTU' #@param {type:'string'}
partition = 'boot' #@param {type:'string'}
version = '' #@param {type:'string'}
package = 'auto' #@param ['auto', 'AP', 'BL', 'CP', 'CSC', 'HOME_CSC', 'USERDATA']
threads = 8 #@param {type:'integer'}


In [ ]:
from pathlib import Path
import subprocess, sys

name = Path(partition).name
if name.lower().endswith('.lz4'):
    name = name[:-4]
if not name.lower().endswith(('.img', '.bin')):
    name += '.img'
output = Path('/content') / name
output.unlink(missing_ok=True)

cmd = [
    sys.executable, '-m', 'twrpdtgen.cloud_slice', 'fetch',
    '-m', model, '-r', region,
    '--partition', partition,
    '--package', package,
    '-j', str(threads),
    '-o', str(output),
    '--colab-download',
]
if version.strip():
    cmd.extend(['-v', version.strip()])

print('Running:', ' '.join(cmd))
subprocess.run(cmd, check=True)
